In [44]:
import sys
import json
import matplotlib.pyplot as plt
import numpy as np
import argparse
import re

from pathlib import Path
from datetime import datetime
from pathlib import Path
from typing import Any, Dict, List
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.colorbar import Colorbar


import optuna
from optuna.study import MaxTrialsCallback
from optuna.trial import Trial
from optuna.pruners import MedianPruner
from optuna.visualization.matplotlib import (
    plot_optimization_history,
    plot_slice,
    plot_param_importances,
)

# Add parent directory to path
try:
    # For scripts
    sys.path.insert(0, str(Path(__file__).parent.parent.parent))
except NameError:
    # For Jupyter notebooks - add the known project root
    sys.path.insert(0, str(Path.cwd().parent.parent))

from ml_events_utils import stylesheet_default
from ml_events_utils import color_gio as color_dict
from ml_events_utils import color_deep as color_dict

import matplotlib.pyplot as plt
import numpy as np

from run_hyperparam_scan_optuna import create_study_summary_figure, apply_log_scale_to_y_axes

In [2]:
arguments = {
    "settings_file": "run_settings_scan.yaml",
    "study_name": "pol_scan",
    "storage": "sqlite:///../optimisation_runs/run_large/pol_scan.db",
    "output_root": "scan_runs",
    "plot_only": True
}

arg = argparse.Namespace(**arguments)

print("Arguments:")
for key, value in vars(arg).items():
    print(f"  {key}: {value}")

Arguments:
  settings_file: run_settings_scan.yaml
  study_name: pol_scan
  storage: sqlite:///../optimisation_runs/run_large/pol_scan.db
  output_root: scan_runs
  plot_only: True


## Define plotting routine

In [49]:
def generate_optimization_plots(study: optuna.Study,
                                output_root: Path,
                                study_name: str,
                                storage: str) -> Path:
    """Generate and save optimization analysis plots to a multipage PDF.

    Uses matplotlib-based Optuna visualization functions for direct integration.

    Args:
        study: Loaded Optuna study object
        output_root: Directory to save PDF
        study_name: Name of the study
        storage: Storage URL/path

    Returns:
        Path to generated PDF file
    """
    plt.style.use(stylesheet_default)

    pdf_path = output_root / f"optimization_analysis_{study_name}.pdf"

    # Save all figures to multipage PDF
    with PdfPages(str(pdf_path)) as pdf:
        d = pdf.infodict()
        d["Title"] = f"Optuna Study Analysis: {study_name}"
        d["Author"] = "Optuna"
        d["Subject"] = "Hyperparameter Optimization Analysis"
        d["Keywords"] = "Optuna, Hyperparameter Optimization"
        d["CreationDate"] = datetime.now()

        target_name="Validation Loss"

        # Page 1: Study summary
        fig_summary = create_study_summary_figure(study, study_name, storage, target_name=target_name)
        pdf.savefig(fig_summary, bbox_inches="tight")
        plt.close(fig_summary)

        # Page 2: Optimization history
        try:
            ax = plot_optimization_history(study, target_name=target_name)
            fig_history = plt.gcf()  # Get current figure
            apply_log_scale_to_y_axes(fig_history)  # Apply log scale to objective values
            ax.set_ylim(ymax=1e-3)  # Set upper limit for better visualization of differences
            ax.set_ylim(ymin=2e-4)  # Set lower limit for better visualization of differences
            ax.set_xlim(xmin=-10)  # Start x-axis at 0 for better visualization of early trials

            ylabel = ax.get_ylabel()
            ax.set_ylabel(f"{ylabel.lower()}")
            xlabel = ax.get_xlabel()
            ax.set_xlabel(f"{xlabel.lower()}")

            legend = ax.get_legend().get_texts()
            legend_labels = [text.get_text().lower() for text in legend]
            print(f"Legend texts: {legend_labels}")
            ax.legend(labels=legend_labels, loc="upper right")

            ax.set_title("")  # Clear default title set by plot_optimization_history

            pdf.savefig(fig_history, bbox_inches="tight")
            plt.close(fig_history)
        except Exception as e:
            print(f"Warning: Could not generate optimization history plot: {e}")

        # Page 3: Slice plot (parameter importance via slices)
        try:
            axs = plot_slice(study, target_name=target_name)
            fig = plt.gcf()  # Get current figure (may have multiple subplots)

            for ax in axs:
                ax.set_yscale("log")  # Set log scale for y-axis of each subplot
                ax.set_ylim(ymax=1e-3)  # Set upper limit for better visualization of differences
                ax.set_ylim(ymin=2e-4)  # Set lower limit for better visualization of differences

            axs[0].set_xlabel("batch size")
            axs[1].set_xlabel("learning rate")
            labels = [label.get_text() for label in axs[2].get_xticklabels()]
            labels[-1] = r"$Z$+jet"
            axs[2].set_xticklabels(labels, rotation=45, ha='right', va='top', rotation_mode='anchor')

            labels = [label.get_text() for label in axs[4].get_xticklabels()]
            labels[-1] = r"cross section"
            axs[4].set_xticklabels(labels)

            # fig.set_title("")  # Clear default title set by plot_slice
            ylabel = axs[0].get_ylabel()
            axs[0].set_ylabel(f"{ylabel.lower()}")
            fig.suptitle("")  # Clear any default suptitle set by plot_slice
            # apply_log_scale_to_y_axes(fig)  # Apply log scale to objective values

            label = fig.axes[-1].get_ylabel()
            fig.axes[-1].set_ylabel(f"{label.lower()}")
            pdf.savefig(fig, bbox_inches="tight")
            plt.close(fig)
        except Exception as e:
            print(f"Warning: Could not generate slice plot: {e}")

        # Page 4: Parameter importance ranking (only if enough trials)
        if len(study.trials) >= 2:
            try:
                ax = plot_param_importances(study, target_name=target_name)
                fig_importance = plt.gcf()  # Get current figure
                # Note: importance plot usually has categorical x-axis, log scale not needed
                ax.set_xlabel(ax.get_xlabel().lower())
                ax.set_ylabel(ax.get_ylabel().lower())

                legend = ax.get_legend().get_texts()
                legend_labels = [text.get_text().lower() for text in legend]
                ax.legend(labels=legend_labels)

                labels = [re.sub(r"_", " ", label.get_text()) for label in ax.get_yticklabels()]
                ax.set_yticklabels(labels)

                ax.set_title("")  # Clear default title set by plot_param_importances
                fig_importance.suptitle("")
                pdf.savefig(fig_importance, bbox_inches="tight")
                plt.close(fig_importance)
            except Exception as e:
                print(f"Warning: Could not generate parameter importance plot: {e}")

    return pdf_path

In [50]:
# Handle plot-only mode: load existing study and generate plots
try:
    study = optuna.load_study(study_name=arg.study_name, storage=arg.storage)
except Exception as e:
    print(f"Error loading study '{arg.study_name}' from storage '{arg.storage}': {e}")

# Print study summary to console
best = study.best_trial
print(f"Study: {arg.study_name}")
print(f"Total trials: {len(study.trials)}")
print(f"Best trial number: {best.number}")
print(f"Best objective value: {best.value}")
print(f"Best parameters: {best.params}\n")

# Generate plots
print("Generating optimization analysis plots...")
print(type(study.trials))
valid_trials = []
for i, trial in enumerate(study.trials):
    # print(f"Trial {trial.number}: Value={trial.value}, Params={trial.params}, State={trial.state}")
    if trial.value == None:
        print(f"Warning: Trial {trial.number} has a None value.")
    elif trial.value > 1e15:
        print(f"Warning: Trial {trial.number} has an unusually high value of {trial.value}")
    else:
        valid_trials.append(trial)

try:
    pdf_path = generate_optimization_plots(study, Path.cwd(), arg.study_name, arg.storage)
    print(f"Plots saved to: {pdf_path}")
except Exception as e:
    print(f"Error generating plots: {e}")




Study: pol_scan
Total trials: 961
Best trial number: 501
Best objective value: 0.00022768646150687208
Best parameters: {'batch_size': 2048, 'learning_rate': 0.0031959142469131104, 'optimizer': 'AdamW', 'patience': 35, 'penalties': 'cross_section'}

Generating optimization analysis plots...
<class 'list'>


/tmp/ipykernel_87250/2169419159.py:40: ExperimentalWarning: optuna.visualization.matplotlib._optimization_history.plot_optimization_history is experimental (supported from v2.2.0). The interface can change in the future.
  ax = plot_optimization_history(study, target_name=target_name)


Legend texts: ['validation loss', 'best value']


/tmp/ipykernel_87250/2169419159.py:66: ExperimentalWarning: optuna.visualization.matplotlib._slice.plot_slice is experimental (supported from v2.2.0). The interface can change in the future.
  axs = plot_slice(study, target_name=target_name)
[W 2026-05-06 10:31:26,442] Trial 10 is omitted in visualization because its objective value is inf or nan.
/tmp/ipykernel_87250/2169419159.py:78: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axs[2].set_xticklabels(labels, rotation=45, ha='right', va='top', rotation_mode='anchor')
/tmp/ipykernel_87250/2169419159.py:82: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  axs[4].set_xticklabels(labels)
/tmp/ipykernel_87250/2169419159.py:100: ExperimentalWarning: optuna.visualization.matplotlib._param_importances.plot_param_importances is experimental (supported from v2.2.0). The interface c

Plots saved to: /ptmp/mpp/jlinder/ML_Giovanni/polarisation_tagging/final_results/plots/optimization_analysis_pol_scan.pdf


## Results of the first large fit

In [13]:
print("Best parameters found:")
for key, value in study.best_params.items():
    print(f"  {key}: {value}")

Best parameters found:
  batch_size: 2048
  learning_rate: 0.0031959142469131104
  optimizer: AdamW
  patience: 35
  penalties: cross_section


- `batch_size` = 2048
- `learning_rate` ~ 0.0032
- `optimizer` = AdamW
- `patience` = 35 (or just $>= 20$)
- `penalty` = doesn't really matter